# wikipedia-vision-decision-bench を Colab で回す（ワンストップ）

型付き判定モデル（choice / yes-no / score）の画像理解ベンチ（データセット v1）を、Colab 上で
**リポジトリ取得 → 重み取得 → CPU 測定 → GPU 測定 → OpenAI API（キーがあれば）→ 集計** まで 1 本で回すノートブックです。
中身の処理は `colab/run_colab.py`（このノートブック無しでも `python colab/run_colab.py --steps setup,images,bench,report ...` で動きます）。

使い方: 上から順にセルを実行します。**設定セルだけ編集**すれば足ります。途中で切れても、同じセルをもう一度実行すれば続きから再開します
（問ごとのログで再開。Drive を使うとセッションをまたいで再開できます）。

**先に読むこと**
- 画像（`images-224px.zip`）は Wikimedia Commons の写真を含むため、リポジトリには入っていません。データセットの Release を公開したら **IMAGES_URL** に URL を入れてください（未公開の間は、手元の zip を Drive に置いて **IMAGES_ZIP** で指す）。
- モデルのリポジトリが未公開・Release 未作成のものは、**スキップして次へ進み**、最後に一覧で表示します（全モデルの公開リポジトリ・Release（v0.0.0）が揃うまでの間は、取得できないモデルがここに並びます）。
- **Colab での CPU の時間は参考値**です。時間の採用ゲート（GLiNER の較正 600 ms 以下）は作者の PC で決めた値で、Colab の CPU は別の機械です。較正値は記録しますが、PC の表と並べて比較しないでください。
- 全モデル × 全問 × CPU を 12 時間以内に終えることはできません（作者の PC の値からの見積もりは「計画」セルに出ます）。まず `SAMPLE_N`（サンプル数）で回してください。
- 使うのは Colab Secrets（左のカギアイコン）だけです: `GITHUB_TOKEN`（非公開リポジトリ用）と `OPENAI_API_KEY`（OpenAI 用）。値は表示も保存もしません。
- このノートブックは実際の Colab ではまだ一度も実行していません（作成した PC には Colab がありません）。構文・計画表示・画像の展開と照合・制御の流れ（擬似実行）はローカルで検証済みです。

## (a) ランタイムの確認
GPU で測るには、メニューの「ランタイム → ランタイムのタイプを変更」で **T4 GPU** を選びます（無料枠の T4 は VRAM 約 16 GB）。GPU が無くても CPU と OpenAI は動きます。

In [ ]:
!nvidia-smi
import os, platform, shutil
print("python", platform.python_version(), "| CPU", os.cpu_count(), "| ディスク空き %.0f GB" % (shutil.disk_usage("/content").free / 1024**3 if os.path.isdir("/content") else 0))

## 設定（ここだけ編集）
`MODELS` がモデルの一覧です（追加するときは、adapters の REGISTRY と `colab/run_colab.py` の `SPECS` に 1 件足してから、ここに名前を足します）。

In [ ]:
#@title 設定
MODELS = ["jevision", "onejev", "onejev4", "intern", "strands", "clef", "qev", "d3edge", "d3lite", "d3nano", "openai"]  #@param {type:"raw"}
DEVICES = "auto"          #@param ["auto", "cpu", "cuda", "cpu,cuda"]
LANGS = "en,ja"           #@param {type:"string"}
SETS = "photo,biz,rsn"    #@param {type:"string"}
MOD = "image"             #@param ["image", "text", "all"]
SAMPLE_N = 300            #@param {type:"integer"}
SMOKE_N = 25              #@param {type:"integer"}
THREADS = 0               #@param {type:"integer"}
SKIP_MISSING_PASSAGES = False   #@param {type:"boolean"}
WEIGHTS = "auto"          #@param ["auto", "keep", "purge"]
TIME_BUDGET_H = 11        #@param {type:"number"}
# DEVICES: auto = CPU に加えて GPU があれば cuda も。 SETS: txt / all は Wikipedia 本文が要る（SKIP_MISSING_PASSAGES で本文の無い問を除く）
# SAMPLE_N: セット x 言語ごとの問数（0 = 全問。全モデルで同じ問が選ばれる）。 SMOKE_N: スモーク（デバイス確認 + 簡易正答率）の問数
# THREADS: 0 = min(8, CPU 数)（実際の値は結果に記録）。 WEIGHTS: purge = 取得 -> 測定 -> 重み削除 を 1 モデルずつ（ディスク上限対策）、auto = 足りないときだけ
# TIME_BUDGET_H: これを超えたら新しいブロックを始めない（無料枠は 12 時間で切れる）

# --- リポジトリ ---
OWNER = "Kazuhito00"
RELEASE_TAG = "v0.0.0"   # 各 download_model.py の --tag（全リポジトリの最初の Release。"" = 各スクリプトの既定 = 最新）
# 共有の取得: onejev + onejev4 は OneJev-ONNX-CPU の 1 回（約 13 GB）、d3edge + d3lite + d3nano は D3-ONNX-CPU の 1 回（約 16 GB）。d3-mini は対象外
BENCH_REPO = "https://github.com/Kazuhito00/wikipedia-vision-decision-bench"
BENCH_REF = ""            # ブランチ / タグ（空 = 既定）
BENCH_DIR = "/content/wikipedia-vision-decision-bench"

# --- 画像（データセット v1 の images-224px.zip）---
IMAGES_URL = ""           # Release の URL（公開前は空）
IMAGES_ZIP = ""           # Drive 等に置いた zip のパス（URL の代わり）
IMAGES_SHA256 = ""        # 空 = run_colab.py の既定（2026-10-11 のビルド）

# --- Drive（ログ・結果・画像 zip・重みのキャッシュ）---
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/wikipedia-vision-decision-bench-colab"
DRIVE_CACHE_WEIGHTS = True    # False: Drive にはログと結果だけ
MODEL_DIRS = {}               # 例: {"qev": "/content/drive/MyDrive/models/QEV/onnx"}   取得済み（Drive 等）の重みを使うとき。通常は不要（download_model.py が取る）
EXTRA_ARGS = []               # run_colab.py にそのまま渡す追加引数（例 ["--no-v1-report", "--force-models"]）

## Colab Secrets（任意）
- `GITHUB_TOKEN`: 非公開のリポジトリを clone するときだけ。トークンは URL に入れず、git のヘッダでだけ渡します。**注意**: 各モデルの `download_model.py` には認証が無いので、非公開の Release の重みは取れません（その場合は Drive に重みを置いて `MODEL_DIRS` で指す）。
- `OPENAI_API_KEY`: 設定されているときだけ OpenAI Decisions API を測ります（**有料**）。未設定なら理由を表示してスキップします。

In [ ]:
import os
def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:          # Colab 以外 / 未設定 / ノートブックにアクセスを許可していない
        return None
for _n in ("GITHUB_TOKEN", "OPENAI_API_KEY"):
    _v = _secret(_n)
    if _v:
        os.environ[_n] = _v
    print(_n, "設定あり（値は表示しません）" if os.environ.get(_n) else "未設定")
_v = None

## Drive をマウント（USE_DRIVE のとき）

In [ ]:
import os
DRIVE_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        DRIVE_OK = True
        print("Drive:", DRIVE_DIR)
    except Exception as e:
        print("Drive を使えません（続行）:", type(e).__name__)

## (b) bench を clone して実行用の関数を用意

In [ ]:
import base64, os, signal, subprocess, sys

def git_env():
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    tok = os.environ.get("GITHUB_TOKEN")
    if tok:   # URL にも .git/config にも入れない
        b = base64.b64encode(("x-access-token:" + tok).encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraHeader", GIT_CONFIG_VALUE_0="Authorization: Basic " + b)
    return env

if not os.path.isdir(os.path.join(BENCH_DIR, ".git")):
    cmd = ["git", "clone", "--depth", "1"] + (["--branch", BENCH_REF] if BENCH_REF else []) + [BENCH_REPO, BENCH_DIR]
    r = subprocess.run(cmd, env=git_env(), capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()[-600:]
    if os.environ.get("GITHUB_TOKEN"):
        out = out.replace(os.environ["GITHUB_TOKEN"], "***")
    print(out)
    if r.returncode != 0:
        raise SystemExit("clone に失敗: 非公開なら GITHUB_TOKEN を Secrets に、未公開なら zip を展開して BENCH_DIR を合わせる")
os.chdir(BENCH_DIR)
assert os.path.exists("colab/run_colab.py"), "このリポジトリ（ref）に colab/ がありません"

def base_args():
    a = ["--models", ",".join(MODELS), "--devices", DEVICES, "--langs", LANGS, "--sets", SETS, "--mod", MOD,
         "--sample", str(SAMPLE_N), "--smoke", str(SMOKE_N), "--threads", str(THREADS), "--weights", WEIGHTS,
         "--time-budget-h", str(TIME_BUDGET_H), "--owner", OWNER, "--release-tag", RELEASE_TAG]
    if SKIP_MISSING_PASSAGES: a.append("--skip-missing-passages")
    if IMAGES_URL: a += ["--images-url", IMAGES_URL]
    if IMAGES_ZIP: a += ["--images-zip", IMAGES_ZIP]
    if IMAGES_SHA256: a += ["--images-sha256", IMAGES_SHA256]
    if DRIVE_OK: a += ["--drive", DRIVE_DIR] + ([] if DRIVE_CACHE_WEIGHTS else ["--no-drive-cache"])
    for k, v in MODEL_DIRS.items(): a += ["--model-dir", "%s=%s" % (k, v)]
    return a + list(EXTRA_ARGS)

def cli(steps, *extra):
    cmd = [sys.executable, "-u", "colab/run_colab.py", "--steps", steps] + base_args() + list(extra)
    # 自分のセッション（プロセスグループ）で起動する。セルを止めたときに子も確実に止めるため（止めないと再実行で 2 つが同じログに書く）
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in p.stdout:
            print(line, end="")
        return p.wait()
    except BaseException:          # KeyboardInterrupt（セルの停止）など
        try:
            os.killpg(p.pid, signal.SIGTERM)      # PID（グループ）指定。名前での kill はしない
        except OSError:
            pass
        raise

## 計画（ネットワーク・GPU 不要）
モデルごとのリポジトリ・ディスク・RAM/VRAM の下限・時間の見積もりを表示します。時間は作者の PC（CPU 8 スレッド）の 1 問あたり中央値 × 問数で、**Colab の CPU は別の機械です**。`?` は各リポジトリに記載が無い値です。

In [ ]:
cli("plan", "--dry-run")

## (c) 依存のインストール（uv）
`uv pip install --system`（uv が無ければ、uv を入れるためだけに pip を 1 回使います）。pillow / numpy、GPU があれば **onnxruntime-gpu**（無ければ onnxruntime。両方は併存できず、同じフォルダを上書きするので、片方を外してから入れます）。
CUDA の組み合わせ: 各モデルの README は onnxruntime-gpu 1.31 + CUDA 13 + cuDNN 9 で書かれています。Colab のドライバが CUDA 12 系のときは `onnxruntime-gpu>=1.20,<1.24`（CUDA 12 版の想定。**未検証**）を入れ、`ORT_GPU_SPEC` 環境変数で上書きできます。
入れたあと CUDA の小さなセッションを実際に作って、CUDA が使えるかを確認します。使えなければ cuda デバイスは全モデルでスキップされます（CPU は影響なし）。

In [ ]:
# 例: os.environ["ORT_GPU_SPEC"] = "onnxruntime-gpu==1.22.0"
cli("env,setup")

## (d) 画像データ
`IMAGES_ZIP` → Drive キャッシュ → `IMAGES_URL`（Release）の順に探し、zip の sha256 を照合してから展開し、全問の画像が存在することと、224px キャッシュのファイル名が衝突しないことを確かめます。

In [ ]:
rc = cli("images")
assert rc == 0, "画像が用意できていません（上のメッセージを確認）"

## (e)(f) 重みの取得・較正・スモーク・本番（ワンストップ）
モデルごとに **clone → 重み取得 → デバイスごとに「スモーク → 本番ブロック」→（`WEIGHTS` が purge、または auto でディスクが足りないとき）重み削除** を順に行います。
- スモーク: 少数の問で、**実際に使われたプロバイダ**（`providers_used`）を見ます。CUDA を頼んだのに CPU になっているモデル、`got=NA` が半分を超える（メモリ不足・上限超え）モデルは、**そのモデルのそのデバイスだけスキップ**して次へ進みます。
- 文書化された RAM / VRAM の下限（例: clef の重み 19 GB は T4 の VRAM 16 GB に載らない）を超えるものは、実行せずスキップして理由を表示します。
- CPU ブロックの前後に GLiNER2.5-Decide の較正（参考値）を測り、結果と `blocks.log` に記録します（取得できなければ較正なしで続行）。
- 10 分ごとにログと結果を Drive に保存します。中断したら **このセルをもう一度実行**してください。
- 全部終わると、モデル x デバイスごとの状態（ok / skipped: 理由）の一覧を表示します。

In [ ]:
rc = cli("bench")
print("exit", rc)

## (g) 集計と HTML
`results/runs/v1/*.json|csv` から、ブロック別・グループ別・形式別の表（markdown / csv / html）と、状態の一覧を作り、結果の zip にまとめます。
さらに `report/v1_report.py` の HTML（ダッシュボード）を `colab/make_v1_report.py` 経由で作ります。v1_report.py は 1 回の試行（jevision CPU/GPU と openai）専用の作りなので、このラッパーが一時フォルダに入力（`dist/v1` 相当の問題ファイル、ブロックの結合、全モデルが答えた問だけの比較、`MODELS` の差し替え）を作ってから呼びます（`report/` は変更しません。画像は埋め込みません）。止めるには `EXTRA_ARGS = ["--no-v1-report"]`。

In [ ]:
cli("report")

In [ ]:
# 結果の zip をダウンロード（Drive を使っていれば Drive にも保存済み）
p = open("work/colab/last_bundle.txt").read().strip()
print(p)
try:
    from google.colab import files
    files.download(p)
except Exception as e:
    print("ダウンロードは Colab 上でのみ:", type(e).__name__)

In [ ]:
# 集計表をそのまま見る
from IPython.display import Markdown, display
display(Markdown(open("work/colab/out/colab_summary.md", encoding="utf-8").read()))

## (h) 後片付け
最終の Drive 保存を行い、必要なら重み・clone を消し、ランタイムを解放します（課金枠の節約）。

In [ ]:
import shutil
cli("sync")
FREE_DISK = False           # True: /content のモデルのリポジトリ（重み）を消す。Drive のキャッシュは残る
RELEASE_RUNTIME = False     # True: ランタイムを切断する（保存してから）
if FREE_DISK:
    shutil.rmtree("/content/model_repos", ignore_errors=True)
if RELEASE_RUNTIME:
    from google.colab import runtime
    runtime.unassign()